## Environment

In [1]:
from datetime import datetime
from pathlib import Path
import os
import sys

import pandas as pd

NOTE_DIR = Path.cwd().resolve().parents[3] / 'note'
REPO_ROOT = Path('/Users/xinc/GitHub/Quant')
sys.path.extend([str(NOTE_DIR), str(REPO_ROOT), os.getcwd()])

%load_ext autoreload
%autoreload 2

from module.get_info_FinMind import FinMindClient
from module.get_info_TradingView import Interval, TradingViewClient
from module.get_info_TWSE import GetInfoTWSE
from module.get_info_Finlab import FinlabClient
from module.options.option_tools import compute_iv
from analyzer import StrategyConfig, TXAnalyzer
from cloud_data import TW_OPTIONS_SETTLE_TXO

START = '2020-01-01'
END = datetime.now().strftime('%Y-%m-%d')

TRAIN_RATIO = 0.7
VALIDATION_RATIO = 0.05
SETTLEMENT_PATH = TW_OPTIONS_SETTLE_TXO

tv = TradingViewClient()
fm = FinMindClient()
twse = GetInfoTWSE()
fl = FinlabClient()

## Base Market Data

In [2]:
fm.initialize_frame(stock_id='TX', start_time=START, end_time=END)
analyzer = TXAnalyzer(fm.get_future_price())
analyzer.session_alignment_report()

split_dates = TXAnalyzer.split_periods(
    analyzer.display_df().index,
    start=START,
    train_ratio=TRAIN_RATIO,
    validation_ratio=VALIDATION_RATIO,
)
TRAIN_END = split_dates['train_end']
VALIDATION_START = split_dates['validation_start']
VALIDATION_END = split_dates['validation_end']
TEST_START = split_dates['test_start']
pd.Series(split_dates, name='date')

[LOCAL] finmind/future_price -> /Users/xinc/GitHub/google_drive/Data/trading/tw_futures/TX.csv


train_end          2024-08-02
validation_start   2024-08-05
validation_end     2024-11-25
test_start         2024-11-26
Name: date, dtype: datetime64[ns]

## Factor Discovery

### Price and Calendar

#### 全部因子

In [12]:
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.daily_ret()
training_analyzer.monthly_ret(mode='benchmark')
training_analyzer.indicator_weekday_stats()
training_analyzer.indicator_ma_divergence(window=25)
training_analyzer.indicator_hist_vol(window=40)
training_analyzer.indicator_night_ret()
training_analyzer.indicator_gap_days(after_holiday=False)
training_analyzer.indicator_gap_days(after_holiday=True)

#### 深檢（日盤） ma_divergence X hist_vol

In [13]:
hist_vol_20d = training_analyzer.indicator_hist_vol(window=20, return_series=True)
ma_divergence_factors = {
    f'{window}MA divergence': training_analyzer.indicator_ma_divergence(window=window, return_series=True)
    for window in range(5, 51)
}

report = training_analyzer.compare_factor_by_condition(
    factors=ma_divergence_factors,
    condition_factor=hist_vol_20d,
    condition_name='HistVol 20D',
    return_column='daily_ret',
    condition_bins=5,
    factor_bin_percentile=15,
    factor_value_format='.2%',
    condition_value_format='.2%',
    title='日盤：MA divergence x HistVol 20D',
)

ma25_divergence = training_analyzer.indicator_ma_divergence(window=25, return_series=True)
volatility_windows = range(5, 51)
volatility_report = training_analyzer.compare_factor_by_condition(
    factors={f'{window}D': ma25_divergence for window in volatility_windows},
    condition_factor={
        f'{window}D': training_analyzer.indicator_hist_vol(window=window, return_series=True)
        for window in volatility_windows
    },
    condition_name='HistVol window (day return)',
    return_column='daily_ret',
    condition_bins=5,
    factor_bin_percentile=15,
    factor_value_format='.2%',
    condition_value_format='.2%',
    title='日盤：25MA divergence x HistVol window',
)

In [15]:
overlap_events = training_analyzer.show_divergence_signal_overlap([
    {
        'ma_window': 25,
        'volatility_window': 21,
        'volatility_regime': 4,
        'volatility_bins': 5,
        'divergence_percentile': 15,
    },
    {
        'ma_window': 25,
        'volatility_window': 40,
        'volatility_regime': 4,
        'volatility_bins': 5,
        'divergence_percentile': 15,
    },
])

In [14]:
hist_vol = training_analyzer.indicator_hist_vol(40, return_series=True)
ma_divergence = training_analyzer.indicator_ma_divergence(25, return_series=True)

result = training_analyzer.conditional_factor_sort(
    conditions=[
        {
            "factor": hist_vol,
            "percentile_range": (60, 80),
            "name": "HistVol 40D",
        },
    ],
    sort_factor=ma_divergence,
    bin_percentile=2,
    plot_mode='bin',
    return_column="daily_ret",
    title="日盤：HistVol Q4、低 Fear & Greed 下的 25MA 乖離",
)
display(result.attrs["selection_summary"])

signal_events = training_analyzer.show_divergence_signal_timeline(
    ma_window=25,
    volatility_window=40,
    volatility_regime=4,
    volatility_bins=5,
    divergence_percentile=15,
)

,factor,percentile_range,lower_cutoff,upper_cutoff,observations_before,observations_after
step,,,,,,
1,HistVol 40D,60% to 80%,0.007402,0.008922,1028,206


#### 深檢（夜盤） ma_divergence x hist_vol

In [16]:
hist_vol_40d = training_analyzer.indicator_hist_vol(window=40, return_series=True)
night_ma_divergence_factors = {
    f'{window}MA divergence': training_analyzer.indicator_ma_divergence(window=window, return_series=True)
    for window in range(5, 51, 5)
}

night_report = training_analyzer.compare_factor_by_condition(
    factors=night_ma_divergence_factors,
    condition_factor=hist_vol_40d,
    condition_name='HistVol 40D (day return)',
    return_column='daily_ret_a',
    condition_bins=5,
    factor_bin_percentile=10,
    factor_value_format='.2%',
    condition_value_format='.2%',
    title='夜盤：MA divergence x 日盤 HistVol 40D',
)

ma25_divergence = training_analyzer.indicator_ma_divergence(window=25, return_series=True)
volatility_windows = range(5, 51)
night_volatility_report = training_analyzer.compare_factor_by_condition(
    factors={f'{window}D': ma25_divergence for window in volatility_windows},
    condition_factor={
        f'{window}D': training_analyzer.indicator_hist_vol(window=window, return_series=True)
        for window in volatility_windows
    },
    condition_name='HistVol window (day return)',
    return_column='daily_ret_a',
    condition_bins=5,
    factor_bin_percentile=15,
    factor_value_format='.2%',
    condition_value_format='.2%',
    title='夜盤：25MA divergence x HistVol window',
)

In [17]:
hist_vol = training_analyzer.indicator_hist_vol(40, return_series=True)
ma_divergence = training_analyzer.indicator_ma_divergence(30, return_series=True)

result = training_analyzer.conditional_factor_sort(
    conditions=[
        {
            "factor": hist_vol,
            "percentile_range": (60, 80),
            "name": "HistVol 40D",
        },
    ],
    sort_factor=ma_divergence,
    bin_percentile=1,
    plot_mode='bin',
    return_column="daily_ret_a",
    title="夜盤：HistVol Q4 下的 25MA 乖離",
)
display(result.attrs["selection_summary"])

night_signal_events = training_analyzer.show_divergence_signal_timeline(
    ma_window=30,
    volatility_window=40,
    volatility_regime=4,
    volatility_bins=5,
    divergence_percentile=5,
    return_column='daily_ret_a',
    volatility_return_column='daily_ret',
)

,factor,percentile_range,lower_cutoff,upper_cutoff,observations_before,observations_after
step,,,,,,
1,HistVol 40D,60% to 80%,0.007402,0.008921,1030,206


#### 深檢（日盤）night_ret x hist_vol

In [37]:
hist_vol_40d = training_analyzer.indicator_hist_vol(window=40, return_series=True)
night_ret_factor = training_analyzer.indicator_night_ret(return_series=True)

night_ret_report = training_analyzer.compare_factor_by_condition(
    factors={'night_ret divergence': night_ret_factor},
    condition_factor=hist_vol_40d,
    condition_name='HistVol 40D (day return)',
    return_column='daily_ret',
    condition_bins=5,
    factor_bin_percentile=10,
    factor_value_format='.2%',
    condition_value_format='.2%',
    title='日盤：night_ret divergence x HistVol 40D',
)
night_ret_report['median'].to_csv('test.csv')

night_ret_result = training_analyzer.conditional_factor_sort(
    conditions=[
        {
        'factor': hist_vol_40d,
        'percentile_range': (60, 80),
        'name': 'HistVol 40D',
        },
    ],
    sort_factor=night_ret_factor,
    bin_percentile=5,
    plot_mode='bin',
    return_column='daily_ret',
    title='日盤：HistVol Q4 下的 night_ret divergence',
)
display(night_ret_result.attrs['selection_summary'])

night_ret_signal_events = training_analyzer.show_divergence_signal_timeline(
    factor=night_ret_factor,
    factor_name='night_ret divergence',
    volatility_window=40,
    volatility_regime=1,
    volatility_bins=1,
    divergence_percentile=20,
    return_column='daily_ret',
    volatility_return_column='daily_ret',
)

,factor,percentile_range,lower_cutoff,upper_cutoff,observations_before,observations_after
step,,,,,,
1,HistVol 40D,60% to 80%,0.007402,0.008922,1028,206


In [ ]:
night_ret_report = training_analyzer.conditional_factor_sort(
    conditions=[
        {
            'factor': night_ret_factor,
            'percentile_range': (6, 21),
        }
    ],
    sort_factor=hist_vol_40d,
    return_column='daily_ret'
)

#### get threshold

In [51]:
night_divergence = training_analyzer.indicator_ma_divergence(window=30, return_series=True)
day_divergence = training_analyzer.indicator_ma_divergence(window=25, return_series=True)
hist_vol = training_analyzer.indicator_hist_vol(window=40, return_series=True)

# ma_divergence x hist_vol 日盤
display(
    training_analyzer.double_sort_thresholds(
        primary=hist_vol,
        secondary=day_divergence,
        primary_percentile_range=(60, 80),
        secondary_percentile=15
    )
)

# ma_divergence x hist_vol 夜盤
display(
    training_analyzer.double_sort_thresholds(
    primary=hist_vol,
    secondary=night_divergence,
    primary_percentile_range=(60, 80),
    secondary_percentile=5
    )
)

primary_lower_percentile        60.000000
primary_upper_percentile        80.000000
primary_lower_cutoff             0.007402
primary_upper_cutoff             0.008921
secondary_lower_percentile       0.000000
secondary_upper_percentile      15.000000
secondary_lower_cutoff          -0.075944
secondary_upper_cutoff          -0.025187
observations                  1030.000000
selected_observations          206.000000
Name: double_sort_thresholds, dtype: float64

primary_lower_percentile        60.000000
primary_upper_percentile        80.000000
primary_lower_cutoff             0.007402
primary_upper_cutoff             0.008921
secondary_lower_percentile       0.000000
secondary_upper_percentile       5.000000
secondary_lower_cutoff          -0.078004
secondary_upper_cutoff          -0.053845
observations                  1030.000000
selected_observations          206.000000
Name: double_sort_thresholds, dtype: float64

### Option Positioning

In [ ]:
raw_day = fm.option_institution_position('TXO', start_date=START, end_date=END).copy()
raw_night = twse.get_institution_option_position(
    trading_session='night',
    start_time=START,
    end_time=END,
).copy()
analyzer.add_option_signals(raw_day, raw_night)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_opt_position(
    indicator='Foreign_Opt_Signal',
    trading_session='day',
    sub_analysis=False,
)

### Option Implied Volatility

In [ ]:
option_daily = TXAnalyzer.fetch_option_daily(
    fm,
    option_id='TXO',
    start_date=START,
    end_date=END,
)
settlement_df = pd.read_csv(SETTLEMENT_PATH)
analyzer.add_option_iv_skew(
    option_daily,
    settlement_df,
    iv_calculator=compute_iv,
    risk_free_rate=0.015,
)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_option_iv(trading_session='day')

### Sentiment

In [ ]:
historical_fear_greed = fm.get_cnn_fear_greed(START, END)
latest_fear_greed = TXAnalyzer.fetch_fear_greed()
analyzer.add_fear_greed(historical_fear_greed, latest_fear_greed)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_fear_greed(trading_session='day')
training_analyzer.indicator_fear_greed(trading_session='night')

### Global Market Factors

In [ ]:
move = tv.get_hist(
    symbol='MOVE',
    exchange='TVC',
    interval=Interval.in_daily,
    n_bars=3000,
)
analyzer.add_market_ohlc(move, 'MOVE')
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_move(trading_session='day')
training_analyzer.indicator_move(trading_session='night')
training_analyzer.indicator_move(trading_session='day', sub_analysis=True)

sox = tv.get_hist(
    symbol='SOX',
    exchange='TVC',
    interval=Interval.in_daily,
    n_bars=3000,
)
analyzer.add_market_ohlc(sox, 'SOX')
training_analyzer.indicator_sox(trading_session='day')

### Rates and Margin

In [4]:
df = pd.read_parquet('/Users/xinc/GitHub/google_drive/Data/trading/tw_stock/margin/margin_utilization.parquet')
df

symbol,0015,0050,0051,0052,0053,0054,0055,0056,0057,0058,...,9944,9945,9946,9949,9950,9951,9955,9958,9960,9962
date,,,,,,,,,,,,,,,,,,,,,
2009-01-05,1.69,11.27,12.67,0.90,3.77,2.16,13.02,2.32,0.67,0.28,...,5.56,13.13,NaN,10.00,NaN,0.52,14.50,0.91,NaN,12.34
2009-01-06,1.70,10.95,12.66,0.84,3.59,2.19,13.19,2.23,0.67,0.28,...,6.02,13.23,NaN,9.80,NaN,0.52,14.51,0.86,NaN,12.37
2009-01-07,1.70,10.32,12.55,0.83,3.93,2.17,12.96,2.12,0.67,0.22,...,6.25,12.87,NaN,9.79,NaN,0.52,15.49,1.43,NaN,12.53
2009-01-08,1.70,10.45,12.59,0.83,3.76,2.18,13.06,2.13,0.67,0.22,...,6.11,12.79,NaN,9.73,NaN,0.52,15.20,1.82,NaN,12.48
2009-01-09,1.70,10.50,12.57,0.83,3.73,2.15,13.10,2.13,0.67,0.22,...,6.31,12.63,NaN,9.65,NaN,0.51,15.00,2.83,NaN,12.41
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-12-24,NaN,1.41,0.13,0.03,0.31,0.04,0.57,0.94,0.00,0.00,...,24.04,4.24,0.70,23.42,15.96,10.45,5.31,36.35,64.91,5.40
2018-12-25,NaN,1.47,0.12,0.03,0.31,0.04,0.54,0.93,0.00,0.00,...,24.04,4.25,0.69,23.46,16.16,10.52,5.30,36.21,64.91,5.41
2018-12-26,NaN,1.46,0.12,0.03,0.31,0.04,0.53,0.94,0.00,0.00,...,24.04,4.30,0.69,23.45,16.21,10.56,5.30,35.98,64.93,5.41


In [6]:
margin_balance = fl.get_data(
    'margin_balance:融資券總餘額',
    start_date=START,
    end_date=END,
)
margin_features = (
    margin_balance[['上市融資交易金額', '上櫃融資交易金額']]
    .apply(pd.to_numeric, errors='coerce')
    .sum(axis=1, min_count=1)
    .rename('MarginPurchaseMoney')
    .to_frame()
)
analyzer.merge_features(margin_features)
training_analyzer = analyzer.for_period(end=TRAIN_END)
training_analyzer.indicator_margin_delta()

Daily usage: 75.1 / 500 MB - margin_balance:融資券總餘額
[REMOTE 2020-01-01..2026-07-21] finlab/margin_balance:融資券總餘額 -> /Users/xinc/GitHub/google_drive/Data/trading/tw_stock/margin/margin_balance.parquet
[DATA GAP] finlab/margin_balance:融資券總餘額 | requested=2020-01-01..2026-07-21 | api_rows=0 | available=none | unresolved=yes | stored=/Users/xinc/GitHub/google_drive/Data/trading/tw_stock/margin/margin_balance.parquet


RuntimeError: FinLab dataset 'margin_balance:融資券總餘額' did not cover requested range 2020-01-01..2026-07-21; returned range=none; stored range=none; path=/Users/xinc/GitHub/google_drive/Data/trading/tw_stock/margin/margin_balance.parquet

## Research Data Check

確認這次實際跑過的研究資料是否已合併、是否有缺值，以及最後更新日。

In [ ]:
analyzer.for_period(end=TRAIN_END).feature_status([
    'MOVE_open',
    'SOX_open',
    'Foreign_Opt_Signal_a',
    'SkewSlope',
    'fear_greed',
    'US_bond_5y',
])

## Hypothesis and Position Design

只有通過探索、且能說明交易理由的關係才放進設定。這裡建立基準假設與要驗證的候選部位。

In [ ]:
baseline_config = StrategyConfig()

candidate_config = StrategyConfig(
    day_long_position=0.5,
    day_short_position=-0.5,
    night_position=0.0,
)

## Validation

只用 validation 期間比較已經提出的設定，選定一個設定後就不要再依 validation 或 test 結果修改它。


In [ ]:
baseline_validation = analyzer.evaluate(
    baseline_config,
    start=VALIDATION_START,
    end=VALIDATION_END,
)
candidate_validation = analyzer.evaluate(
    candidate_config,
    start=VALIDATION_START,
    end=VALIDATION_END,
)

validation_summary = pd.concat(
    {
        'baseline': analyzer.summarize_result(baseline_validation),
        'candidate': analyzer.summarize_result(candidate_validation),
    },
    axis=1,
)
validation_summary


In [ ]:
validation_comparison = pd.DataFrame({
    'baseline': baseline_validation['strat_ret'],
    'candidate': candidate_validation['strat_ret'],
}).dropna()
validation_comparison.cumsum().plot(title='Validation: Baseline vs Candidate')


## Test

選定設定後才執行這段。Test 期間只用來確認固定策略的表現，不再調整設定。


In [ ]:
selected_config = baseline_config

test_result = analyzer.evaluate(selected_config, start=TEST_START)
test_summary = analyzer.summarize_result(test_result)
test_summary

# 確認 test 後，再將同一份固定設定寫入正式結果檔。
analyzer.backtest(config=selected_config, point_version=False, start=TEST_START)